# FieldAI: Receipt Information Extraction with Layout-Aware Transformers

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Disha-01-alt/field-ai/blob/main/notebooks/FieldAI_Complete_Colab_Benchmark.ipynb)

## Project Overview & Research Angle
This notebook trains and benchmarks three model paradigms on the **ICDAR 2019 SROIE** dataset:
1. **Baseline 1:** Image Preprocessing (OpenCV) + PaddleOCR + Advanced Spatial Regex Extractor
2. **Baseline 2:** `bert-base-uncased` Token Classification (1D Text Only)
3. **Multimodal Model:** `microsoft/layoutlm-base-uncased` Token Classification (Text + 2D Spatial Layout)

> **Core Research Question:**  
> $$\Delta F1_{\text{field}} = F1_{\text{LayoutLM}}(\text{field}) - F1_{\text{BERT}}(\text{field})$$
> *How much does 2D spatial arrangement improve extraction for Company, Date, and Total over text alone?*

### New Extraction Pipeline (used by Baseline 1 and the Live Demo)
```
Receipt Image -> Preprocessing (OpenCV) -> PaddleOCR -> Text + Bounding Boxes -> Advanced Regex -> Company, Date, Total
```
- **Preprocessing** (`src/preprocessing.py`): resize, grayscale, denoise, CLAHE contrast boost, deskew.
- **PaddleOCR** (`src/paddle_ocr_engine.py`): finds text lines and reads them, giving text + boxes scaled to $[0, 1000]$.
- **Advanced Regex** (`src/advanced_regex.py`): groups boxes into lines and uses position + patterns to pick Company, Date and Total.
- `src/pipeline.py` (`ReceiptExtractionPipeline`) runs all of these steps in one call.

### How the benchmark is run (real photos, all of them)
Every receipt photo of SROIE (500 train + 126 validation + 347 test) is read by the pipeline above. The three methods are then trained and tested on what PaddleOCR actually reads, not on the dataset's ready-made text:
- **Train:** BERT and LayoutLM are trained on the OCR text of the train + validation photos (so they learn to cope with OCR mistakes).
- **Test:** all three methods are scored on the OCR text of the 347 test photos, which no model has seen.
- OCR is slow, so every photo is read once and saved (optionally to Google Drive); reruns and re-scoring are instant.

---  
### Step 1: Verify Hardware Acceleration (GPU)

In [1]:
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: Running on CPU. For fast training, enable GPU via Runtime > Change runtime type > T4 GPU.")

PyTorch Version: 2.14.1+cu130
CUDA Available: True
GPU Device: Tesla T4


---
### Step 2: Clone Repository & Install Dependencies
Installs the project requirements, including **PaddleOCR** and **OpenCV** for the new extraction pipeline.

In [ ]:
# If running inside Colab, clone the project repository
import os
from pathlib import Path

if not Path("src").exists():
    !git clone https://github.com/Disha-01-alt/field-ai.git
    %cd field-ai

# Installs PaddleOCR 3.x + OpenCV for the extraction pipeline (see requirements.txt)
!pip install -q -r requirements.txt
print("\nDependencies installed and workspace configured!")

---  
### Step 3: Inspect the Preprocessed SROIE Dataset
The dataset contains **500 training receipts** and **126 test receipts** with coordinates normalized to $[0, 1000]$ and BIO entity labels.

*(This is the format reference. Steps 4b-6 build the same format from real OCR output and train on that.)*

In [3]:
import json

with open("data/processed/train.json", "r") as f:
    train_data = json.load(f)
with open("data/processed/test.json", "r") as f:
    test_data = json.load(f)

print(f"Training Receipts: {len(train_data)}")
print(f"Testing Receipts: {len(test_data)}")

sample = test_data[0]
print("\n--- Sample Test Document ---")
print(f"ID: {sample['id']}")
print(f"Target Entities: {sample['entities']}")
print("First 5 words with normalized 2D bboxes:")
for w, b, tag in zip(sample['words'][:5], sample['bboxes'][:5], sample['bio_tags'][:5]):
    print(f"  {w:<15} | BBox: {str(b):<20} | Tag: {tag}")

Training Receipts: 500
Testing Receipts: 126

--- Sample Test Document ---
ID: 369
Target Entities: {'company': 'GARDENIA BAKERIES (KL) SDN BHD', 'date': '14/10/2017', 'address': 'LOT 3, JALAN PELABUR 23/1, 40300 SHAH ALAM, SELANGOR.', 'total': '33.80'}
First 5 words with normalized 2D bboxes:
  GARDENIA        | BBox: [34, 103, 590, 123]  | Tag: B-COMPANY
  BAKERIES        | BBox: [34, 103, 590, 123]  | Tag: I-COMPANY
  (KL)            | BBox: [34, 103, 590, 123]  | Tag: I-COMPANY
  SDN             | BBox: [34, 103, 590, 123]  | Tag: I-COMPANY
  BHD             | BBox: [34, 103, 590, 123]  | Tag: I-COMPANY


---
### Step 4: Read ALL Receipt Photos (Preprocessing + PaddleOCR) and Score the Rules
This step uses the **real receipt photos** of SROIE, not the ready-made words and boxes from the JSON files:

```
receipt photo -> preprocessing (OpenCV) -> PaddleOCR -> text + boxes -> advanced regex -> {company, date, total}
```
1. Downloads the SROIE test photos and their answers from Hugging Face.
2. Runs the whole pipeline on every photo (`src/real_eval.py`). OCR is the slow part (about 15-20 s per photo on Colab, **roughly 5 hours for all 973 photos**), so every result is **saved**. If Colab disconnects, run the cell again and it continues where it stopped.
3. Scores the rules on the test photos and explains every mistake: did OCR misread the text, or did the rules pick the wrong line?

**Keep the results on Google Drive** (`USE_DRIVE = True`) so they survive a disconnect and a new session.
Set `NUM_IMAGES` to a small number (for example `20`) for a quick trial; `0` means all photos.

In [ ]:
# Keep the OCR results on Google Drive so a disconnected Colab session can continue where it stopped.
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE_ROOT = "/content/drive/MyDrive/field-ai/ocr_cache"
else:
    CACHE_ROOT = "outputs/ocr_cache"

NUM_IMAGES = 0       # photos per split: 0 = ALL (500 train + 126 val + 347 test). Try 20 for a quick trial.
FAST_OCR = True      # small PaddleOCR models: ~3-4x faster, similar quality on receipts

from src.real_eval import evaluate_real_data

regex_results = evaluate_real_data(
    split="test",
    limit=NUM_IMAGES or None,
    fast=FAST_OCR,
    cache_root=CACHE_ROOT,
    output_path="results/regex_baseline.json",
)

---
### Step 4b: Build the Training and Test Data from the OCR of All Photos
The train and validation photos are read the same way (the saved results are reused), and the dataset's answers are matched to the OCR words to make word labels. A word PaddleOCR misread by a letter is still matched (for example `GERPANG` for `GERBANG`).

- `data/ocr/train.json`: train + validation photos (626)
- `data/ocr/test.json`: test photos (347), never used for training

Same format as `data/processed/*.json`, so the training code in Steps 5 and 6 is unchanged. The printout shows how often each answer could be found in the OCR text: a field that is often *not found* is one OCR reads badly.

In [ ]:
from src.ocr_dataset import prepare_ocr_dataset

ocr_summary = prepare_ocr_dataset(
    cache_root=CACHE_ROOT,
    fast=FAST_OCR,
    limit=NUM_IMAGES or None,
)

# Steps 5 and 6 train and test on the OCR text of the photos
TRAIN_FILE, TEST_FILE = "data/ocr/train.json", "data/ocr/test.json"
# (to repeat the old clean-text benchmark instead, use data/processed/train.json and data/processed/test.json)

---
### Step 5: Fine-Tune Model 2 — BERT-base (Text-Only Transformer)
Fine-tunes `bert-base-uncased` for token classification using 1D sequential WordPiece text tokens (without spatial coordinates), trained on the **OCR text of the train photos** and tested on the OCR text of the **test photos**.

In [ ]:
from src.train_bert import train_bert

# 5 epochs on ~626 samples takes a few minutes on a Colab T4 GPU
bert_results = train_bert(
    train_file=TRAIN_FILE,
    test_file=TEST_FILE,
    output_path="results/bert_results.json",
    epochs=5,
    batch_size=8,
    lr=3e-5,
)

---
### Step 6: Fine-Tune Model 3 — LayoutLM-base (Multimodal: Text + 2D Layout)
Fine-tunes `microsoft/layoutlm-base-uncased` using WordPiece text tokens and the 2D boxes $[x_0, y_0, x_1, y_1]$ that PaddleOCR found, trained on the **OCR of the train photos** and tested on the OCR of the **test photos**.

In [ ]:
from src.train_layoutlm import train

# Trains LayoutLM with early stopping and saves the best checkpoint to
# outputs/layoutlm/best_model (~10-15 min on a Colab T4). Config (epochs,
# batch size, LR, patience) lives at the top of src/train_layoutlm.py.
# It trains on TRAIN_FILE (OCR text of the train photos) and is tested on TEST_FILE.
train(train_file=TRAIN_FILE, test_file=TEST_FILE, output_dir="outputs/layoutlm")

# --- Evaluate the trained model with the SAME harness as regex & BERT so the
#     three-way comparison and delta-F1 are apples-to-apples (metrics.py). ---
import json, os, torch
from transformers import LayoutLMForTokenClassification, LayoutLMTokenizerFast
from src.inference import predict_fields
from src.metrics import compute_field_metrics

model = LayoutLMForTokenClassification.from_pretrained("outputs/layoutlm/best_model")
tok   = LayoutLMTokenizerFast.from_pretrained("outputs/layoutlm/best_model")
model.to("cuda" if torch.cuda.is_available() else "cpu")

test_data = json.load(open(TEST_FILE))
predictions, ground_truths = {}, {}
for doc in test_data:
    predictions[doc["id"]]   = predict_fields(model, tok, doc["words"], doc["bboxes"])
    ground_truths[doc["id"]] = doc["entities"]

metrics = compute_field_metrics(predictions, ground_truths)
layoutlm_results = {
    "model": "LayoutLM-base (Multimodal)",
    "modality": "Text + 2D Bounding Boxes",
    "metrics": metrics,
    "sample_predictions": {k: predictions[k] for k in list(predictions)[:5]},
}
os.makedirs("results", exist_ok=True)
with open("results/layoutlm_results.json", "w") as f:
    json.dump(layoutlm_results, f, indent=2)

print("\nLayoutLM test Macro F1:", round(metrics["macro_avg"]["f1"], 4))
print("Wrote results/layoutlm_results.json")

---
### Step 6b: Real Photos — Rules vs LayoutLM vs Hybrid
The **same OCR output of the test photos** from Step 4 (read from the saved results, so no OCR is run again) is given to three extractors:
- **Advanced Regex** — the rules
- **LayoutLM** — the model trained in Step 6 on OCR text
- **Hybrid** — LayoutLM first for company and date, rules first for total (it falls back to the other when the first finds nothing)

Uses the same `NUM_IMAGES`, `FAST_OCR` and `CACHE_ROOT` as Step 4.

In [ ]:
from src.real_eval import compare_methods

# `model` and `tok` are the trained LayoutLM from Step 6. The OCR of the test photos is read from the saved results.
real_comparison = compare_methods(
    split="test",
    limit=NUM_IMAGES or None,
    fast=FAST_OCR,
    cache_root=CACHE_ROOT,
    model=model,
    tokenizer=tok,
)

---  
### Step 7: Final Benchmark Table & Field-Wise $\Delta F1$ Visualizations

In [ ]:
import json
from src.metrics import format_results_markdown_table, compute_delta_f1
from src.visualize import generate_svg_chart, generate_matplotlib_plot

with open("results/regex_baseline.json") as f:
    regex_m = json.load(f)
with open("results/bert_results.json") as f:
    bert_m = json.load(f)
with open("results/layoutlm_results.json") as f:
    layout_m = json.load(f)

all_models = {
    "PaddleOCR + Advanced Regex": regex_m,
    "BERT-base (Text Only)": bert_m,
    "LayoutLM-base (Multimodal)": layout_m,
}

# Print Markdown Summary Table
table_md = format_results_markdown_table(all_models)
print("### Final Benchmark Results Table:\n")
print(table_md)

# Compute and Print Empirical Delta
deltas = compute_delta_f1(bert_m["metrics"], layout_m["metrics"])
print("\n### Empirical ΔF1 = F1(LayoutLM) - F1(BERT):")
for field, val in deltas.items():
    print(f"  {field.upper():<12}: {val * 100:+.2f}%")

# Generate Visualization Plots
benchmark_chart_data = {
    model_name: {k: v["f1"] for k, v in data["metrics"].items() if "f1" in v}
    for model_name, data in all_models.items()
}
generate_svg_chart(benchmark_chart_data, "results/f1_comparison.svg")
generate_matplotlib_plot(benchmark_chart_data, "results/f1_comparison.png")

---  
### Step 8: View the Generated Benchmark Plot Inline

In [ ]:
from IPython.display import Image, display
if os.path.exists("results/f1_comparison.png"):
    display(Image(filename="results/f1_comparison.png"))
else:
    print("Chart saved as results/f1_comparison.svg")

---
## Live Demo — Extract Fields from a Receipt Image

```
receipt.jpg -> Preprocessing (OpenCV) -> PaddleOCR -> Text + Bounding Boxes -> Advanced Regex -> {company, date, total}
                                                                            \-> LayoutLM      -> {company, date, total}
```

The same PaddleOCR text + boxes feed **both** extractors, so you can compare the rule-based result with the trained model.
The training step saves the best model to `outputs/layoutlm/best_model`.

### Who runs what
- **① STUDENT — run ONCE** to publish the model to a public HF repo (cell a). Needs your HF write token (pasted into the Colab prompt).
- **② GRADER — just run cells (b) + (c)** to test on any uploaded receipt. No token, no training — model downloads from the public HF repo.

> **Accuracy note:** PaddleOCR returns one box per text **line**, which matches how SROIE (the training data) was labelled,
> so LayoutLM sees boxes like the ones it was trained on. Results are still best on clean, printed, SROIE-style receipts.

In [ ]:
# =====================================================================
# (a)  STUDENT - RUN ONCE.  GRADERS: SKIP THIS CELL.
# Publishes the trained model (outputs/layoutlm/best_model) to a PUBLIC HF repo.
# =====================================================================
from huggingface_hub import login
from transformers import LayoutLMForTokenClassification, LayoutLMTokenizerFast

# >>> EDIT: replace `your-username` with your Hugging Face username <<<
HF_REPO_ID = "your-username/fieldai-layoutlm-sroie"

login()   # paste your HF WRITE token when prompted (stays in this session)

_m = LayoutLMForTokenClassification.from_pretrained("outputs/layoutlm/best_model")
_t = LayoutLMTokenizerFast.from_pretrained("outputs/layoutlm/best_model")
_m.push_to_hub(HF_REPO_ID)          # public by default
_t.push_to_hub(HF_REPO_ID)
print(f"Published (public): https://huggingface.co/{HF_REPO_ID}")


In [ ]:
# =====================================================================
# (b)  GRADER - RUN THIS.  Builds the extraction pipeline and loads the model.
# =====================================================================
import torch
from transformers import LayoutLMForTokenClassification, LayoutLMTokenizerFast
from src.pipeline import ReceiptExtractionPipeline
from src.inference import predict_fields

# New extraction pipeline: Preprocessing -> PaddleOCR -> Text + Boxes -> Advanced Regex.
extraction_pipeline = ReceiptExtractionPipeline(lang="en", fast=True)

# Grader: set to the student's public HF repo, e.g. "disha-sahu/fieldai-layoutlm-sroie".
# (If you just trained in this same session, "outputs/layoutlm/best_model" also works.)
MODEL_SOURCE = "outputs/layoutlm/best_model"   # or the student's public HF repo id, e.g. "disha-sahu/fieldai-layoutlm-sroie"

infer_model = LayoutLMForTokenClassification.from_pretrained(MODEL_SOURCE)
infer_tok   = LayoutLMTokenizerFast.from_pretrained(MODEL_SOURCE)
infer_model.to("cuda" if torch.cuda.is_available() else "cpu")
print("Pipeline and model ready. Upload a receipt in the next cell.")

In [ ]:
# =====================================================================
# (c)  GRADER - RUN THIS.  Upload a receipt image and read the fields.
# =====================================================================
import cv2
import matplotlib.pyplot as plt
from google.colab import files

uploaded = files.upload()                       # choose a .jpg / .png receipt
path = next(iter(uploaded))

# 1) Preprocessing -> PaddleOCR -> Text + Bounding Boxes -> Advanced Regex
result = extraction_pipeline.process(path)
regex_fields = result["fields"]

# 2) Feed the same PaddleOCR output to LayoutLM.
#    Each PaddleOCR token is a text line; split it into words and give every
#    word its line's box (the same format as data/processed/*.json).
words, boxes = [], []
for tok in result["tokens"]:
    for w in tok["text"].split():
        words.append(w)
        boxes.append(tok["box"])
layoutlm_fields = predict_fields(infer_model, infer_tok, words, boxes) if words else {}

prep = result["preprocessing"]
print(f"\nPaddleOCR found {result['tokens_count']} text boxes in {result['lines_count']} lines "
      f"(scale={prep['scale_factor']:.2f}, skew={prep['skew_angle']:.2f} deg)")
print(f"\nExtracted fields from {path}:")
print(f"  {'FIELD':8}  {'ADVANCED REGEX':35}  LAYOUTLM")
for k in ["company", "date", "total"]:
    print(f"  {k.upper():8}  {(regex_fields.get(k) or '(not found)'):35}  {layoutlm_fields.get(k) or '(not found)'}")

# 3) Show the PaddleOCR boxes (green = company, blue = date, red = total)
vis = ReceiptExtractionPipeline.visualize_result(result["preprocessed_image"], result["tokens"], regex_fields)
plt.figure(figsize=(8, 12))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("PaddleOCR boxes + Advanced Regex fields")
plt.show()